In [16]:
import os
import sys
import numpy as np
import pandas as pd
import librosa
import librosa.effects
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# --- Step 1: Configuration & Parameters ---
LABEL_PATH_1_2 = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
AUDIO_DIR_1_2 = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
MODEL_SAVE_PATH_1_2 = "best_copd_1_2_model.keras"

N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 128, 150, 25, 32
NOISE_FACTOR, TIME_SHIFT_MAX_SEC, PITCH_SHIFT_STEPS, TIME_STRETCH_RATE = 0.005, 0.2, 4, 0.8
INITIAL_LEARNING_RATE = 0.001


# --- Step 2: Helper Functions ---
def add_gaussian_noise(y, noise_factor=NOISE_FACTOR):
    return y + noise_factor * np.random.randn(len(y))

def time_shift(y, sr, shift_max_sec=TIME_SHIFT_MAX_SEC):
    return np.roll(y, int(sr*np.random.uniform(-shift_max_sec, shift_max_sec)))

def extract_log_mel_spectrogram(y, sr):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel_spec)
    if log_mel.shape[1] < MAX_LEN:
        log_mel = np.pad(log_mel, ((0, 0), (0, MAX_LEN - log_mel.shape[1])), mode='constant')
    else:
        log_mel = log_mel[:, :MAX_LEN]
    return log_mel


# --- Step 3: Load Data with Final Validation ---
print("--- Step 3: Loading and Validating Data for COPD 1 vs 2 ---")
df = pd.read_excel(LABEL_PATH_1_2)
df_copd_1_2 = df[df["Diagnosis"].isin(["COPD1", "COPD2"])].copy()

if df_copd_1_2['Diagnosis'].nunique() < 2:
    raise ValueError("The Excel file must contain patients from both 'COPD1' and 'COPD2' diagnoses.")

df_copd_1_2['label_encoded'] = df_copd_1_2['Diagnosis'].apply(lambda x: 0 if x == 'COPD1' else 1)
label_dict_1_2 = dict(zip(df_copd_1_2["Patient ID"], df_copd_1_2["label_encoded"]))
patient_ids_1_2 = list(label_dict_1_2.keys())
patient_labels_1_2 = list(label_dict_1_2.values())

print("\nPerforming stratified patient-aware split...")
try:
    train_pids, test_pids, _, _ = train_test_split(
        patient_ids_1_2, patient_labels_1_2,
        test_size=0.25, random_state=42, stratify=patient_labels_1_2
    )
except ValueError as e:
    print(f"\nFATAL ERROR during train/test split: {e}")
    print("This usually means one class has only 1 patient, making stratification impossible.")
    raise

print(f"Total Patients: {len(patient_ids_1_2)}, Training PIDs: {len(train_pids)}, Testing PIDs: {len(test_pids)}")

X_train, y_train, X_test, y_test = [], [], [], []
for pid in patient_ids_1_2:
    label = label_dict_1_2[pid]
    is_training_patient = pid in train_pids
    for side in ['L', 'R']:
        for i in range(1, 7):
            fname = f"{pid}_{side}{i}.wav"
            fpath = os.path.join(AUDIO_DIR_1_2, fname)
            if not os.path.exists(fpath): continue
            try:
                y_audio, sr = librosa.load(fpath, sr=None)
                spec = extract_log_mel_spectrogram(y_audio, sr)
                if is_training_patient:
                    y_train.extend([label] * 5)
                    X_train.extend([
                        spec, extract_log_mel_spectrogram(add_gaussian_noise(y_audio, sr), sr),
                        extract_log_mel_spectrogram(time_shift(y_audio, sr), sr),
                        extract_log_mel_spectrogram(librosa.effects.pitch_shift(y=y_audio, sr=sr, n_steps=4), sr),
                        extract_log_mel_spectrogram(librosa.effects.time_stretch(y=y_audio, rate=1/TIME_STRETCH_RATE), sr)
                    ])
                elif pid in test_pids:
                    X_test.append(spec)
                    y_test.append(label)
            except Exception as e:
                print(f"Warning: Error processing {fname}: {e}")

X_train, y_train = np.array(X_train), np.array(y_train)
X_test, y_test = np.array(X_test), np.array(y_test)

# --- FINAL VALIDATION BLOCK (The definitive fix) ---
if len(np.unique(y_train)) < 2:
    raise ValueError(
        "\n\nFATAL ERROR: The training set ('y_train') ended up with only one class.\n"
        "This happens because audio files for the other class are MISSING from the directory.\n\n"
        f"Training was attempted with these Patient IDs: {train_pids}\n\n"
        "ACTION REQUIRED: Please verify that the audio files for ALL of these patients actually exist in this folder:\n"
        f"-> {AUDIO_DIR_1_2}\n"
    )

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]
print("\n--- Final Data Shapes ---\n", f"X_train: {X_train.shape}, y_train: {y_train.shape}\n", f"X_test: {X_test.shape}, y_test: {y_test.shape}")

# --- Step 4: Calculate Class Weights for 1 vs 2 ---
print("\n--- Step 4: Handling Class Imbalance for 1 vs 2 ---")
class_weights_array = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weights_1_2 = dict(enumerate(class_weights_array))
print(f"Counts: COPD1(0)={np.sum(y_train == 0)}, COPD2(1)={np.sum(y_train == 1)}")
print(f"Calculated Weights: {class_weights_1_2}")


# --- Step 5: Build the CNN Model ---
print("\n--- Step 5: Building the CNN Model for 1-2 Progression ---")
model_1_2 = Sequential([
    tf.keras.Input(shape=(N_MELS, MAX_LEN, 1)),
    Conv2D(32, (3, 3), activation='relu', padding='same'), BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), Dropout(0.3),
    Conv2D(64, (3, 3), activation='relu', padding='same'), BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), Dropout(0.3),
    Conv2D(128, (3, 3), activation='relu', padding='same'), BatchNormalization(),
    MaxPooling2D(pool_size=(2, 2)), Dropout(0.3),
    Flatten(), Dense(128, activation='relu'), Dropout(0.5),
    Dense(1, activation='sigmoid')
])
model_1_2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model_1_2.summary()


--- Step 3: Loading and Validating Data for COPD 1 vs 2 ---

Performing stratified patient-aware split...
Total Patients: 12, Training PIDs: 9, Testing PIDs: 3

--- Final Data Shapes ---
 X_train: (540, 128, 150, 1), y_train: (540,)
 X_test: (36, 128, 150, 1), y_test: (36,)

--- Step 4: Handling Class Imbalance for 1 vs 2 ---
Counts: COPD1(0)=240, COPD2(1)=300
Calculated Weights: {0: np.float64(1.125), 1: np.float64(0.9)}

--- Step 5: Building the CNN Model for 1-2 Progression ---


2025-06-30 16:03:25.904082: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 128, 150, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 128, 150, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 64, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64, 75, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 64, 75, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64, 75, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 32, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32, 37, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 32, 37, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 32, 37, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 16, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 16, 18, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 36864)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │     4,718,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,812,417 (18.36 MB)

 Trainable params: 4,811,969 (18.36 MB)

 Non-trainable params: 448 (1.75 KB)

In [17]:
# --- Step 6: Train the 1-2 Progression Model ---
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
model_checkpoint = ModelCheckpoint(MODEL_SAVE_PATH_1_2, save_best_only=True, monitor='val_accuracy', verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)
print(f"\n--- Step 6: Starting Model Training for COPD 1-2 ---\n")
history_1_2 = model_1_2.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS, batch_size=BATCH_SIZE,
    callbacks=[early_stop, model_checkpoint, reduce_lr],
    class_weight=class_weights_1_2
)


--- Step 6: Starting Model Training for COPD 1-2 ---

Epoch 1/25


2025-06-30 16:03:39.613513: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 78643200 exceeds 10% of free system memory.
2025-06-30 16:03:39.664811: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 78643200 exceeds 10% of free system memory.
2025-06-30 16:03:39.706835: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 78643200 exceeds 10% of free system memory.
2025-06-30 16:03:40.371572: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 78643200 exceeds 10% of free system memory.
2025-06-30 16:03:40.371627: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 78643200 exceeds 10% of free system memory.


17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 778ms/step - accuracy: 0.5472 - loss: 10.4778
Epoch 1: val_accuracy improved from -inf to 0.33333, saving model to best_copd_1_2_model.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 18s 813ms/step - accuracy: 0.5463 - loss: 10.3755 - val_accuracy: 0.3333 - val_loss: 5.3143 - learning_rate: 0.0010
Epoch 2/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 739ms/step - accuracy: 0.4847 - loss: 3.8411
Epoch 2: val_accuracy improved from 0.33333 to 0.66667, saving model to best_copd_1_2_model.keras
17/17 ━━━━━━━━━━━━━━━━━━━━ 13s 766ms/step - accuracy: 0.4870 - loss: 3.8005 - val_accuracy: 0.6667 - val_loss: 1.0821 - learning_rate: 0.0010
Epoch 3/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 742ms/step - accuracy: 0.4820 - loss: 1.7950
Epoch 3: val_accuracy did not improve from 0.66667
17/17 ━━━━━━━━━━━━━━━━━━━━ 13s 754ms/step - accuracy: 0.4826 - loss: 1.7864 - val_accuracy: 0.6667 - val_loss: 0.8145 - learning_rate: 0.0010
Epoch 4/25
17/17 ━━━━━━━━━━━━━━━━━━━━ 0s 745ms/step - accuracy: 0.4926 - loss: 0.

In [18]:
# --- Step 7: Evaluate the Final 1-2 Model ---
print("\n--- Step 7: Evaluating Best Saved 1-2 Model ---")
model_1_2.load_weights(MODEL_SAVE_PATH_1_2)
loss, accuracy = model_1_2.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy (1 vs 2): {accuracy*100:.2f}%")
print(f"Final Test Loss (1 vs 2): {loss:.4f}")


--- Step 7: Evaluating Best Saved 1-2 Model ---

Final Test Accuracy (1 vs 2): 66.67%
Final Test Loss (1 vs 2): 1.0821


In [20]:
# --- Step 8: Patient-Level Analysis (Averaging Method) ---
def run_patient_analysis_by_average(trained_model, labels_path, audio_dir, target_diagnosis, sort_ascending):
    print(f"\n--- 📈 Starting Analysis for '{target_diagnosis}' Patients (Averaging Scores) ---")
    df_labels = pd.read_excel(labels_path)
    patient_ids = df_labels[df_labels['Diagnosis'] == target_diagnosis]['Patient ID'].tolist()
    if not patient_ids: return None
    print(f"Found {len(patient_ids)} patients. Analyzing files...")
    results_list = []
    for pid in patient_ids:
        scores = []
        for side in ['L', 'R']:
            for i in range(1, 7):
                fpath = os.path.join(audio_dir, f"{pid}_{side}{i}.wav")
                if not os.path.exists(fpath): continue
                try:
                    y, sr = librosa.load(fpath, sr=None)
                    spec = extract_log_mel_spectrogram(y, sr)
                    prob = trained_model.predict(np.expand_dims(spec, axis=(0, -1)), verbose=0)[0][0]
                    scores.append(prob)
                except Exception as e:
                    print(f"Warning: Could not process file {fpath}: {e}")
        if scores:
            results_list.append({'Patient ID': pid, 'Avg_Score': np.mean(scores), 'Audio_Files_Found': len(scores)})
    results_df = pd.DataFrame(results_list).sort_values(by='Avg_Score', ascending=sort_ascending)
    return results_df.reset_index(drop=True)

def assess_copd2_patient(row):
    return "✅ Model Confident (Correctly resembles Stage 2)" if row['Avg_Score'] >= 0.5 else "⚠️ Model Lacks Confidence (False Negative)"

def assess_copd1_progression_risk(row):
    return "⚠️ High Risk (Resembles Stage 2 - False Positive)" if row['Avg_Score'] >= 0.5 else "✅ Low Risk (Correctly identified as Stage 1)"

# --- Analyze COPD2 Patients ---
copd2_avg_df = run_patient_analysis_by_average(model_1_2, LABEL_PATH_1_2, AUDIO_DIR_1_2, 'COPD2', True)
if copd2_avg_df is not None:
    copd2_avg_df['Assessment'] = copd2_avg_df.apply(assess_copd2_patient, axis=1)
    print("\n--- ✅ COPD2 Patient Confidence Results (Averaging Method) ---")
    print("Shows model's confidence in identifying patients known to have Stage 2. Avg_Score >= 0.5 is correct.")
    print(copd2_avg_df.to_string())

# --- Analyze COPD1 Patients ---
copd1_avg_df = run_patient_analysis_by_average(model_1_2, LABEL_PATH_1_2, AUDIO_DIR_1_2, 'COPD1', False)
if copd1_avg_df is not None:
    copd1_avg_df['Assessment'] = copd1_avg_df.apply(assess_copd1_progression_risk, axis=1)
    print("\n--- ⚠️ COPD1 Patient Progression Risk Results (Averaging Method) ---")
    print("Shows which Stage 1 patients are flagged as being at high risk of progressing to Stage 2.")
    print(copd1_avg_df.to_string())


--- 📈 Starting Analysis for 'COPD2' Patients (Averaging Scores) ---
Found 7 patients. Analyzing files...

--- ✅ COPD2 Patient Confidence Results (Averaging Method) ---
Shows model's confidence in identifying patients known to have Stage 2. Avg_Score >= 0.5 is correct.
  Patient ID  Avg_Score  Audio_Files_Found                                       Assessment
0       H042   0.923945                 12  ✅ Model Confident (Correctly resembles Stage 2)
1       H018   0.927974                 12  ✅ Model Confident (Correctly resembles Stage 2)
2       H044   0.942048                 12  ✅ Model Confident (Correctly resembles Stage 2)
3       H031   0.942467                 12  ✅ Model Confident (Correctly resembles Stage 2)
4       H038   0.950762                 12  ✅ Model Confident (Correctly resembles Stage 2)
5       H030   0.957373                 12  ✅ Model Confident (Correctly resembles Stage 2)
6       H028   0.966230                 12  ✅ Model Confident (Correctly resembles Sta

After data balancing

COPD 1 - 5 samples augmented to 6 types

COPD 2 - 7 samples augmented to 5 types

In [39]:
import os
import sys
import numpy as np
import pandas as pd
import librosa
import librosa.effects
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# --- Step 1: Configuration & Parameters ---
LABEL_PATH_1_2 = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
AUDIO_DIR_1_2 = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
MODEL_SAVE_PATH_1_2 = "best_copd_1_2_oversampled_model.keras"

N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 128, 150, 25, 32
NOISE_FACTOR, TIME_SHIFT_MAX_SEC, PITCH_SHIFT_STEPS, TIME_STRETCH_RATE = 0.005, 0.2, 4, 0.8
INITIAL_LEARNING_RATE = 0.001


# --- Step 2: Helper Functions ---
def add_gaussian_noise(y, noise_factor=NOISE_FACTOR): return y + noise_factor * np.random.randn(len(y))
def time_shift(y, sr, shift_max_sec=TIME_SHIFT_MAX_SEC): return np.roll(y, int(sr*np.random.uniform(-shift_max_sec, shift_max_sec)))
def extract_log_mel_spectrogram(y, sr):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel_spec)
    if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0, 0), (0, MAX_LEN - log_mel.shape[1])), mode='constant')
    else: log_mel = log_mel[:, :MAX_LEN]
    return log_mel


# --- Step 3: Load Data and Apply Oversampling to Balance the Dataset ---
print("--- Step 3: Loading Data for COPD 1 vs 2 ---")
df = pd.read_excel(LABEL_PATH_1_2)
df_copd_1_2 = df[df["Diagnosis"].isin(["COPD1", "COPD2"])].copy()

if df_copd_1_2['Diagnosis'].nunique() < 2:
    raise ValueError("The Excel file must contain patients from both 'COPD1' and 'COPD2'.")

df_copd_1_2['label_encoded'] = df_copd_1_2['Diagnosis'].apply(lambda x: 0 if x == 'COPD1' else 1) # COPD1=0, COPD2=1
label_dict_1_2 = dict(zip(df_copd_1_2["Patient ID"], df_copd_1_2["label_encoded"]))
patient_ids_1_2 = list(label_dict_1_2.keys())
patient_labels_1_2 = list(label_dict_1_2.values())

print("\nPerforming stratified patient-aware split...")
try:
    train_pids, test_pids, y_train_pids_labels, _ = train_test_split(
        patient_ids_1_2, patient_labels_1_2,
        test_size=0.25, random_state=42, stratify=patient_labels_1_2
    )
except ValueError as e:
    raise ValueError(f"\nFATAL ERROR: {e}\nThis means a class has only 1 patient.") from e

# --- OVERSAMPLING LOGIC START ---
print("\nCalculating oversampling rate to balance the training set...")
base_augmentations = 5
num_copd1_train = y_train_pids_labels.count(0)
num_copd2_train = y_train_pids_labels.count(1)

# Identify which label belongs to the minority class
if num_copd1_train < num_copd2_train:
    minority_label = 0
    minority_count, majority_count = num_copd1_train, num_copd2_train
else:
    minority_label = 1
    minority_count, majority_count = num_copd2_train, num_copd1_train

# Calculate how many augmentations are needed for the minority class
if minority_count > 0:
    oversampling_factor = majority_count / minority_count
    augmentations_for_minority = round(base_augmentations * oversampling_factor)
else: # Should not happen due to stratify, but good practice
    augmentations_for_minority = base_augmentations

print(f"Training patient distribution: {num_copd1_train} COPD1, {num_copd2_train} COPD2.")
print(f"Each majority class patient will generate {base_augmentations} samples.")
print(f"Each minority class patient will generate {augmentations_for_minority} samples to balance the data.")
# --- OVERSAMPLING LOGIC END ---

X_train, y_train, X_test, y_test = [], [], [], []
for pid in patient_ids_1_2:
    label = label_dict_1_2[pid]
    for side in ['L', 'R']:
        for i in range(1, 7):
            fpath = os.path.join(AUDIO_DIR_1_2, f"{pid}_{side}{i}.wav")
            if not os.path.exists(fpath): continue
            try:
                y_audio, sr = librosa.load(fpath, sr=None)
                if pid in train_pids:
                    num_augmentations = augmentations_for_minority if label == minority_label else base_augmentations
                    y_train.extend([label] * num_augmentations)
                    # Create a list of augmented samples. Add more variations if needed.
                    augs = [
                        extract_log_mel_spectrogram(y_audio, sr),
                        extract_log_mel_spectrogram(add_gaussian_noise(y_audio), sr),
                        extract_log_mel_spectrogram(time_shift(y_audio, sr), sr),
                        extract_log_mel_spectrogram(librosa.effects.pitch_shift(y=y_audio, sr=sr, n_steps=PITCH_SHIFT_STEPS), sr),
                        extract_log_mel_spectrogram(librosa.effects.time_stretch(y=y_audio, rate=1/TIME_STRETCH_RATE), sr),
                        # Add more augmentations to draw from if needed for high oversampling rates
                        extract_log_mel_spectrogram(librosa.effects.pitch_shift(y=y_audio, sr=sr, n_steps=-PITCH_SHIFT_STEPS), sr),
                        extract_log_mel_spectrogram(time_shift(y_audio, sr), sr) # another random time shift
                    ]
                    X_train.extend(augs[:num_augmentations])
                elif pid in test_pids:
                    X_test.append(extract_log_mel_spectrogram(y_audio, sr))
                    y_test.append(label)
            except Exception as e:
                print(f"Warning: Error processing {fpath}: {e}")

X_train, y_train = np.array(X_train), np.array(y_train)
X_test, y_test = np.array(X_test), np.array(y_test)
if len(np.unique(y_train)) < 2: raise ValueError(f"FATAL: Training set has <2 classes due to MISSING audio files in '{AUDIO_DIR_1_2}'.")
X_train, X_test = X_train[..., np.newaxis], X_test[..., np.newaxis]
print("\n--- Final Data Shapes ---\n", f"X_train: {X_train.shape}, y_train: {y_train.shape}\n", f"X_test: {X_test.shape}, y_test: {y_test.shape}")

# --- Step 4: Verify Class Balance ---
print("\n--- Step 4: Verifying Data Balance ---")
print("Since we used oversampling, the training data is now manually balanced.")
print(f"Final training sample counts: COPD1(0)={np.sum(y_train == 0)}, COPD2(1)={np.sum(y_train == 1)}")
print("Class weights are not needed.")


# --- Step 5: Build the CNN Model ---
print("\n--- Step 5: Building the CNN Model for 1-2 Progression ---")
# model_1_2 = Sequential([
#     tf.keras.Input(shape=(N_MELS, MAX_LEN, 1)),
#     Conv2D(32, (3, 3), activation='relu', padding='same'), BatchNormalization(),
#     MaxPooling2D(pool_size=(2, 2)), Dropout(0.3),
#     Conv2D(64, (3, 3), activation='relu', padding='same'), BatchNormalization(),
#     MaxPooling2D(pool_size=(2, 2)), Dropout(0.3),
#     Conv2D(128, (3, 3), activation='relu', padding='same'), BatchNormalization(),
#     MaxPooling2D(pool_size=(2, 2)), Dropout(0.3),
#     Flatten(), Dense(128, activation='relu'), Dropout(0.5),
#     Dense(1, activation='sigmoid')
# ])
# model_1_2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
# model_1_2.summary()

from tensorflow.keras.layers import Input, Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense, Add
from tensorflow.keras.models import Model

def resnet_block(input_tensor, filters):
    """A simple residual block."""
    x = Conv2D(filters, (3, 3), activation='relu', padding='same')(input_tensor)
    x = BatchNormalization()(x)
    x = Conv2D(filters, (3, 3), activation='relu', padding='same')(x)
    x = BatchNormalization()(x)
    
    # This is the "skip connection"
    # It adds the original input back to the output of the convolutional block
    x = Add()([x, input_tensor])
    return x

# --- Define the Model Architecture using the ResNet block ---
input_layer = Input(shape=(N_MELS, MAX_LEN, 1))

# Initial Conv layer to get to the right number of filters (e.g., 64)
x = Conv2D(64, (3, 3), activation='relu', padding='same')(input_layer)
x = BatchNormalization()(x)

# --- Add Residual Blocks ---
x = resnet_block(x, filters=64)
x = resnet_block(x, filters=64)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.3)(x)

x = resnet_block(x, filters=64)
x = resnet_block(x, filters=64)
x = MaxPooling2D(pool_size=(2, 2))(x)
x = Dropout(0.3)(x)

# --- Classifier Head ---
x = Flatten()(x)
x = Dense(128, activation='relu')(x)
x = Dropout(0.5)(x)
output_layer = Dense(1, activation='sigmoid')(x)

# Create the final model
model_1_2_resnet = Model(inputs=input_layer, outputs=output_layer)
model_1_2_resnet.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model_1_2_resnet.summary()

--- Step 3: Loading Data for COPD 1 vs 2 ---

Performing stratified patient-aware split...

Calculating oversampling rate to balance the training set...
Training patient distribution: 4 COPD1, 5 COPD2.
Each majority class patient will generate 5 samples.
Each minority class patient will generate 6 samples to balance the data.

--- Final Data Shapes ---
 X_train: (588, 128, 150, 1), y_train: (588,)
 X_test: (36, 128, 150, 1), y_test: (36,)

--- Step 4: Verifying Data Balance ---
Since we used oversampling, the training data is now manually balanced.
Final training sample counts: COPD1(0)=288, COPD2(1)=300
Class weights are not needed.

--- Step 5: Building the CNN Model for 1-2 Progression ---


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 128, 150,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_15 (Conv2D)  │ (None, 128, 150,  │        640 │ input_layer_3[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_15[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_16 (Conv2D)  │ (None, 128, 150,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_16[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_17 (Conv2D)  │ (None, 128, 150,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_17[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_4 (Add)         │ (None, 128, 150,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_18 (Conv2D)  │ (None, 128, 150,  │     36,928 │ add_4[0][0]       │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_18[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_19 (Conv2D)  │ (None, 128, 150,  │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 150,  │        256 │ conv2d_19[0][0]   │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_5 (Add)         │ (None, 128, 150,  │          0 │ batch_normalizat… │
│                     │ 64)               │            │ add_4[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_8     │ (None, 64, 75,    │          0 │ add_5[0][0]       │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 64, 75,    │          0 │ max_pooling2d_8[… │
│ (Dropout)           │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_20 (Conv2D)  │ (None, 64, 75,    │     36,928 │ dropout_11[0][0]  │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 75,    │        256 │ conv2d_20[0][0] 

 Total params: 9,997,953 (38.14 MB)

 Trainable params: 9,996,801 (38.13 MB)

 Non-trainable params: 1,152 (4.50 KB)

In [36]:
# --- Step 6: Train the Model ---
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
model_checkpoint = ModelCheckpoint(MODEL_SAVE_PATH_1_2, save_best_only=True, monitor='val_accuracy', verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)
print(f"\n--- Step 6: Starting Model Training on Balanced Data ---\n")
# We DO NOT pass class_weight here, as the data is already balanced
history_1_2 = model_1_2.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, model_checkpoint, reduce_lr])


--- Step 6: Starting Model Training on Balanced Data ---

Epoch 1/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 648ms/step - accuracy: 0.6401 - loss: 0.5192
Epoch 1: val_accuracy improved from -inf to 0.41667, saving model to best_copd_1_2_oversampled_model.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 669ms/step - accuracy: 0.6412 - loss: 0.5179 - val_accuracy: 0.4167 - val_loss: 0.8251 - learning_rate: 0.0010
Epoch 2/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 663ms/step - accuracy: 0.7133 - loss: 0.4757
Epoch 2: val_accuracy improved from 0.41667 to 0.44444, saving model to best_copd_1_2_oversampled_model.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 684ms/step - accuracy: 0.7144 - loss: 0.4749 - val_accuracy: 0.4444 - val_loss: 1.0353 - learning_rate: 0.0010
Epoch 3/25
19/19 ━━━━━━━━━━━━━━━━━━━━ 0s 667ms/step - accuracy: 0.8180 - loss: 0.3689
Epoch 3: val_accuracy improved from 0.44444 to 0.55556, saving model to best_copd_1_2_oversampled_model.keras
19/19 ━━━━━━━━━━━━━━━━━━━━ 13s 687ms/step - accuracy: 0.8165 - loss: 0.3

In [37]:
# --- Step 7: Evaluate the Final Model ---
print("\n--- Step 7: Evaluating Best Saved Model ---")
model_1_2.load_weights(MODEL_SAVE_PATH_1_2)
loss, accuracy = model_1_2.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy (1 vs 2): {accuracy*100:.2f}%")
print(f"Final Test Loss (1 vs 2): {loss:.4f}")


--- Step 7: Evaluating Best Saved Model ---

Final Test Accuracy (1 vs 2): 55.56%
Final Test Loss (1 vs 2): 0.9790


In [40]:
# --- Step 8: Patient-Level Analysis (Averaging Method with Detailed Scores) ---

def run_patient_analysis_by_average(trained_model, labels_path, audio_dir, target_diagnosis, sort_ascending):
    """
    Analyzes patients by averaging scores and also shows the raw prediction
    for each of the 12 audio files per patient.
    """
    print(f"\n--- 📈 Starting Analysis for '{target_diagnosis}' Patients (Averaging Scores) ---")
    df_labels = pd.read_excel(labels_path)
    patient_ids = df_labels[df_labels['Diagnosis'] == target_diagnosis]['Patient ID'].tolist()
    if not patient_ids: return None
    print(f"Found {len(patient_ids)} patients. Analyzing files...")
    results_list = []
    for pid in patient_ids:
        scores = []
        for side in ['L', 'R']:
            for i in range(1, 7):
                fpath = os.path.join(audio_dir, f"{pid}_{side}{i}.wav")
                if not os.path.exists(fpath): continue
                try:
                    y, sr = librosa.load(fpath, sr=None)
                    spec = extract_log_mel_spectrogram(y, sr)
                    prob = trained_model.predict(np.expand_dims(spec, axis=(0, -1)), verbose=0)[0][0]
                    scores.append(prob)
                except Exception as e:
                    print(f"Warning: Could not process file {fpath}: {e}")

        # If any scores were successfully collected for the patient:
        if scores:
            # --- ADDITION: Print the detailed scores for this patient ---
            formatted_scores = [f'{s:.4f}' for s in scores]
            print(f"  -> Raw Scores for Patient {pid}: {formatted_scores}")
            
            # Append the calculated average to the results list
            results_list.append({'Patient ID': pid, 'Avg_Score': np.mean(scores), 'Audio_Files_Found': len(scores)})
            
    # If no results were generated at all, exit.
    if not results_list:
        print("Could not generate any analysis results. Check if audio files exist.")
        return None

    results_df = pd.DataFrame(results_list).sort_values(by='Avg_Score', ascending=sort_ascending)
    return results_df.reset_index(drop=True)

# Assessment functions remain the same
def assess_copd2_patient(row):
    return "✅ Model Confident (Correctly resembles Stage 2)" if row['Avg_Score'] >= 0.5 else "⚠️ Model Lacks Confidence (False Negative)"

def assess_copd1_progression_risk(row):
    return "⚠️ High Risk (Resembles Stage 2 - False Positive)" if row['Avg_Score'] >= 0.5 else "✅ Low Risk (Correctly identified as Stage 1)"

# --- Run Analysis for COPD2 Patients ---
copd2_avg_df = run_patient_analysis_by_average(model_1_2, LABEL_PATH_1_2, AUDIO_DIR_1_2, 'COPD2', True)
if copd2_avg_df is not None:
    copd2_avg_df['Assessment'] = copd2_avg_df.apply(assess_copd2_patient, axis=1)
    print("\n--- ✅ COPD2 Patient Confidence Results (Averaging Method) ---")
    print("Shows model's confidence in identifying patients known to have Stage 2. Avg_Score >= 0.5 is correct.")
    print(copd2_avg_df.to_string())

# --- Run Analysis for COPD1 Patients ---
copd1_avg_df = run_patient_analysis_by_average(model_1_2, LABEL_PATH_1_2, AUDIO_DIR_1_2, 'COPD1', False)
if copd1_avg_df is not None:
    copd1_avg_df['Assessment'] = copd1_avg_df.apply(assess_copd1_progression_risk, axis=1)
    print("\n--- ⚠️ COPD1 Patient Progression Risk Results (Averaging Method) ---")
    print("Shows which Stage 1 patients are flagged as being at high risk of progressing to Stage 2.")
    print(copd1_avg_df.to_string())


--- 📈 Starting Analysis for 'COPD2' Patients (Averaging Scores) ---
Found 7 patients. Analyzing files...
  -> Raw Scores for Patient H018: ['0.9585', '0.9930', '0.9683', '0.9209', '0.9946', '0.9794', '0.9402', '0.9764', '0.9738', '0.9938', '0.9898', '0.9981']
  -> Raw Scores for Patient H028: ['0.8745', '0.9564', '0.9635', '0.9132', '0.8788', '0.8706', '0.9170', '0.8631', '0.9199', '0.8957', '0.6491', '0.9875']
  -> Raw Scores for Patient H030: ['0.9761', '0.9740', '0.9293', '0.5022', '0.8417', '0.8656', '0.9631', '0.8492', '0.9919', '0.6016', '0.8483', '0.9950']
  -> Raw Scores for Patient H031: ['0.9976', '0.9738', '0.8924', '0.8240', '0.9142', '0.6711', '0.9747', '0.9812', '0.9916', '0.8795', '0.9794', '0.7426']
  -> Raw Scores for Patient H038: ['0.7748', '0.5026', '0.0402', '0.7305', '0.0451', '0.8181', '0.1601', '0.1640', '0.3982', '0.4474', '0.6468', '0.3387']
  -> Raw Scores for Patient H042: ['0.9131', '0.9477', '0.9030', '0.8864', '0.9581', '0.8764', '0.9623', '0.7627', '0.9

In [41]:
# --- Step 8: Patient-Level Analysis (Counting Method with Detailed Scores) ---

def run_patient_analysis_by_vote(trained_model, labels_path, audio_dir, target_diagnosis, sort_ascending):
    """
    Analyzes patients using a majority vote method. It first shows the raw
    prediction for each audio file, then counts the votes to determine the
    final patient-level prediction.
    """
    print(f"\n--- 🗳️ Starting Analysis for '{target_diagnosis}' Patients (Majority Vote) ---")
    df_labels = pd.read_excel(labels_path)
    patient_ids = df_labels[df_labels['Diagnosis'] == target_diagnosis]['Patient ID'].tolist()
    if not patient_ids: return None
    print(f"Found {len(patient_ids)} patients. Analyzing files and counting votes...")
    
    patient_scores_data = {}
    for pid in patient_ids:
        scores = []
        for side in ['L', 'R']:
            for i in range(1, 7):
                fpath = os.path.join(audio_dir, f"{pid}_{side}{i}.wav")
                if not os.path.exists(fpath): continue
                try:
                    y, sr = librosa.load(fpath, sr=None)
                    spec = extract_log_mel_spectrogram(y, sr)
                    prob = trained_model.predict(np.expand_dims(spec, axis=(0, -1)), verbose=0)[0][0]
                    scores.append(prob)
                except Exception as e:
                    print(f"Warning: Could not process file {fpath}: {e}")
        
        # If scores were collected, show the details and store them
        if scores:
            # --- ADDITION: Print the detailed scores for this patient ---
            formatted_scores = [f'{s:.4f}' for s in scores]
            print(f"  -> Raw Scores for Patient {pid}: {formatted_scores}")
            
            patient_scores_data[pid] = scores

    # If no patient data could be processed, exit.
    if not patient_scores_data:
        print("Could not generate any analysis results. Check if audio files exist.")
        return None
        
    results_list = []
    for pid, scores in patient_scores_data.items():
        # --- COUNTING LOGIC ---
        copd2_votes = sum(1 for s in scores if s >= 0.5)
        copd1_votes = len(scores) - copd2_votes
        
        # Determine final prediction by majority (defaulting to Stage 1 on a tie)
        final_prediction = 'COPD2' if copd2_votes > copd1_votes else 'COPD1'
            
        results_list.append({
            'Patient ID': pid, 'COPD2_Votes': copd2_votes, 'COPD1_Votes': copd1_votes,
            'Total_Files': len(scores), 'Final_Prediction': final_prediction
        })

    results_df = pd.DataFrame(results_list).sort_values(by='COPD2_Votes', ascending=sort_ascending)
    return results_df.reset_index(drop=True)

def assess_prediction_vs_truth(row, true_label):
    """Compares the majority vote prediction to the known truth."""
    if row['Final_Prediction'] == true_label:
        return f"✅ Correct (Predicted {row['Final_Prediction']})"
    else:
        return f"❌ Incorrect (Predicted {row['Final_Prediction']}, but was {true_label})"


# --- Run Analysis on COPD2 Patients (High-Severity Group) ---
copd2_vote_df = run_patient_analysis_by_vote(
    trained_model=model_1_2, labels_path=LABEL_PATH_1_2, audio_dir=AUDIO_DIR_1_2,
    target_diagnosis='COPD2', sort_ascending=True # Show patients with FEWEST votes for COPD2 first
)
if copd2_vote_df is not None:
    copd2_vote_df['Assessment'] = copd2_vote_df.apply(assess_prediction_vs_truth, true_label='COPD2', axis=1)
    print("\n--- ✅ COPD2 Patient Majority Vote Results ---")
    print("This table shows if the model's majority vote matched the patient's actual 'COPD2' diagnosis.")
    print(copd2_vote_df.to_string())

# --- Run Analysis on COPD1 Patients (Progression Risk Group) ---
copd1_vote_df = run_patient_analysis_by_vote(
    trained_model=model_1_2, labels_path=LABEL_PATH_1_2, audio_dir=AUDIO_DIR_1_2,
    target_diagnosis='COPD1', sort_ascending=False # Show Stage 1 patients with MOST votes for COPD2 first
)
if copd1_vote_df is not None:
    copd1_vote_df['Assessment'] = copd1_vote_df.apply(assess_prediction_vs_truth, true_label='COPD1', axis=1)
    print("\n--- ⚠️ COPD1 Patient Progression Risk Results ---")
    print("This table shows if a Stage 1 patient is incorrectly flagged as having progressed to Stage 2.")
    print("Incorrect '❌' assessments here are patients the model thinks are at HIGH RISK of progression.")
    print(copd1_vote_df.to_string())


--- 🗳️ Starting Analysis for 'COPD2' Patients (Majority Vote) ---
Found 7 patients. Analyzing files and counting votes...
  -> Raw Scores for Patient H018: ['0.9585', '0.9930', '0.9683', '0.9209', '0.9946', '0.9794', '0.9402', '0.9764', '0.9738', '0.9938', '0.9898', '0.9981']
  -> Raw Scores for Patient H028: ['0.8745', '0.9564', '0.9635', '0.9132', '0.8788', '0.8706', '0.9170', '0.8631', '0.9199', '0.8957', '0.6491', '0.9875']
  -> Raw Scores for Patient H030: ['0.9761', '0.9740', '0.9293', '0.5022', '0.8417', '0.8656', '0.9631', '0.8492', '0.9919', '0.6016', '0.8483', '0.9950']
  -> Raw Scores for Patient H031: ['0.9976', '0.9738', '0.8924', '0.8240', '0.9142', '0.6711', '0.9747', '0.9812', '0.9916', '0.8795', '0.9794', '0.7426']
  -> Raw Scores for Patient H038: ['0.7748', '0.5026', '0.0402', '0.7305', '0.0451', '0.8181', '0.1601', '0.1640', '0.3982', '0.4474', '0.6468', '0.3387']
  -> Raw Scores for Patient H042: ['0.9131', '0.9477', '0.9030', '0.8864', '0.9581', '0.8764', '0.9623

Checking with the other data

In [1]:
# =================================================================================
#
#       COPD Progression Model: Stage 1 vs. Stage 2 (Oversampling & ResNet)
#       This script will train and save the model as "testing.keras".
#
# =================================================================================

import os
import sys
import numpy as np
import pandas as pd
import librosa
import librosa.effects
import tensorflow as tf
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, BatchNormalization, Add, Activation
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import random

# --- Step 1: Configuration & Parameters ---
LABEL_PATH_1_2 = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
AUDIO_DIR_1_2 = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
# --- Using your new model name ---
MODEL_SAVE_PATH_1_2 = "testing.keras"

N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 20, 150, 25, 32
INITIAL_LEARNING_RATE = 0.001


# --- Step 2: Helper Functions (Corrected with uniform signature) ---
def augment_gaussian_noise(y, sr, noise_factor=0.005):
    return y + noise_factor * np.random.randn(len(y))
def augment_time_shift(y, sr, shift_max_sec=0.2):
    return np.roll(y, int(sr*np.random.uniform(-shift_max_sec, shift_max_sec)))
def augment_pitch_shift(y, sr, n_steps=4):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)
def augment_time_stretch(y, sr, rate=0.9):
    return librosa.effects.time_stretch(y, rate=rate)
def extract_log_mel_spectrogram(y, sr):
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
    log_mel = librosa.power_to_db(mel_spec)
    if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0, 0), (0, MAX_LEN - log_mel.shape[1])), mode='constant')
    else: log_mel = log_mel[:, :MAX_LEN]
    return log_mel
AUGMENTATION_FUNCTIONS = [augment_gaussian_noise, augment_time_shift, augment_pitch_shift, augment_time_stretch]


# --- Step 3: Load Data and Apply Oversampling ---
print("--- Step 3: Loading Data for COPD 1 vs 2 ---")
df = pd.read_excel(LABEL_PATH_1_2)
df_copd_1_2 = df[df["Diagnosis"].isin(["COPD1", "COPD2"])].copy()
if df_copd_1_2['Diagnosis'].nunique() < 2: raise ValueError("Excel must contain patients from both 'COPD1' and 'COPD2'.")

df_copd_1_2['label_encoded'] = df_copd_1_2['Diagnosis'].apply(lambda x: 0 if x == 'COPD1' else 1)
label_dict_1_2 = dict(zip(df_copd_1_2["Patient ID"], df_copd_1_2["label_encoded"]))
patient_ids_1_2, patient_labels_1_2 = list(label_dict_1_2.keys()), list(label_dict_1_2.values())

print("\nPerforming stratified patient-aware split...")
try:
    train_pids, test_pids, y_train_pids_labels, _ = train_test_split(patient_ids_1_2, patient_labels_1_2, test_size=0.25, random_state=42, stratify=patient_labels_1_2)
except ValueError as e: raise ValueError(f"\nFATAL ERROR: {e}\nThis means a class has only 1 patient.") from e

train_files_map = []
for pid in train_pids:
    label = label_dict_1_2[pid]
    paths = [os.path.join(AUDIO_DIR_1_2, f) for f in os.listdir(AUDIO_DIR_1_2) if f.startswith(str(pid)) and f.endswith('.wav')]
    for path in paths:
        if os.path.exists(path):
            train_files_map.append({'path': path, 'label': label})
train_df = pd.DataFrame(train_files_map)

print("\nCalculating oversampling ratios for the training set...")
train_class_counts = train_df['label'].value_counts()
majority_class_count = train_class_counts.max()
augmentation_ratios = (majority_class_count / train_class_counts).round().astype(int) - 1
for i, ratio in augmentation_ratios.items(): print(f"  Class {i}: {ratio} extra augmentations needed per file.")

print("\nGenerating final datasets...")
X_train, y_train, X_test, y_test = [], [], [], []
for _, row in train_df.iterrows():
    try:
        y_audio, sr = librosa.load(row['path'], sr=None)
        label, num_augs = row['label'], augmentation_ratios.get(label, 0)
        X_train.append(extract_log_mel_spectrogram(y_audio, sr)); y_train.append(label)
        for _ in range(num_augs):
            aug_func = random.choice(AUGMENTATION_FUNCTIONS)
            y_augmented = aug_func(y_audio.copy(), sr); X_train.append(extract_log_mel_spectrogram(y_augmented, sr)); y_train.append(label)
    except Exception as e: print(f"Warning: Could not process {row['path']}: {e}")
for pid in test_pids:
    label = label_dict_1_2[pid]
    paths = [os.path.join(AUDIO_DIR_1_2, f) for f in os.listdir(AUDIO_DIR_1_2) if f.startswith(str(pid)) and f.endswith('.wav')]
    for path in paths:
        if os.path.exists(path):
            try:
                y_audio, sr = librosa.load(path, sr=None); X_test.append(extract_log_mel_spectrogram(y_audio, sr)); y_test.append(label)
            except Exception as e: print(f"Warning: Could not process test file {path}: {e}")

# --- Normalization and Reshaping Block ---
X_train_raw = np.array(X_train)
train_mean = np.mean(X_train_raw, axis=0)
train_std = np.std(X_train_raw, axis=0)
print("\nSaving normalization statistics for this model...")
np.save("logmel_testing_mean.npy", train_mean)
np.save("logmel_testing_std.npy", train_std)
X_train = (X_train_raw - train_mean) / (train_std + 1e-6)
X_test = (np.array(X_test) - train_mean) / (train_std + 1e-6)
X_train, X_test = X_train[..., np.newaxis], X_test[..., np.newaxis]
y_train, y_test = np.array(y_train), np.array(y_test)
print("✅ Stats saved. Data prepared.")


# --- Step 4: Build the ResNet-like CNN Model ---
print("\n--- Step 4: Building the ResNet-like CNN Model for 1-2 Progression ---")
from tensorflow.keras.regularizers import l2
def resnet_block(input_tensor, filters):
    x = Conv2D(filters, (3, 3), activation='relu', padding='same')(input_tensor); x = BatchNormalization()(x)
    x = Conv2D(filters, (3, 3), activation='relu', padding='same')(x); x = BatchNormalization()(x)
    return Add()([x, input_tensor])
input_layer = Input(shape=(N_MELS, MAX_LEN, 1))
x = Conv2D(64, (3, 3), activation='relu', padding='same')(input_layer); x = BatchNormalization()(x)
x = resnet_block(x, filters=64); x = resnet_block(x, filters=64)
x = MaxPooling2D(pool_size=(2, 2))(x); x = Dropout(0.3)(x)
x = resnet_block(x, filters=64); x = resnet_block(x, filters=64)
x = MaxPooling2D(pool_size=(2, 2))(x); x = Dropout(0.3)(x)
x = Flatten()(x); x = Dense(128, activation='relu')(x); x = Dropout(0.5)(x)
output_layer = Dense(1, activation='sigmoid')(x)
model_1_2 = Model(inputs=input_layer, outputs=output_layer)
model_1_2.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
model_1_2.summary()


# --- Step 5: Train the Model ---
print(f"\n--- Step 5: Starting Model Training ---\n")
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True, verbose=1)
model_checkpoint = ModelCheckpoint(MODEL_SAVE_PATH_1_2, save_best_only=True, monitor='val_accuracy', verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=1e-6, verbose=1)
history_1_2 = model_1_2.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=[early_stop, model_checkpoint, reduce_lr])


# --- Step 6: Evaluate the Final Model ---
print("\n--- Step 6: Evaluating Best Saved Model ---")
model_1_2.load_weights(MODEL_SAVE_PATH_1_2)
loss, accuracy = model_1_2.evaluate(X_test, y_test, verbose=0)
print(f"\nFinal Test Accuracy (1 vs 2): {accuracy*100:.2f}%")
print(f"Final Test Loss (1 vs 2): {loss:.4f}")
print(f"\n✅ Training complete. Model saved to '{MODEL_SAVE_PATH_1_2}'")

2025-07-13 19:23:11.561195: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 19:23:11.568358: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-07-13 19:23:11.589131: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1752414791.623702    5917 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1752414791.633564    5917 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1752414791.660046    5917 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

--- Step 3: Loading Data for COPD 1 vs 2 ---

Performing stratified patient-aware split...

Calculating oversampling ratios for the training set...
  Class 1: 0 extra augmentations needed per file.
  Class 0: 0 extra augmentations needed per file.

Generating final datasets...

Saving normalization statistics for this model...
✅ Stats saved. Data prepared.

--- Step 4: Building the ResNet-like CNN Model for 1-2 Progression ---


2025-07-13 19:23:18.754126: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 20, 150,   │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 20, 150,   │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 20, 150,   │        256 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 20, 150,   │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 150,   │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 20, 150,   │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 150,   │        256 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 20, 150,   │          0 │ batch_normalizat… │
│                     │ 64)               │            │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 20, 150,   │     36,928 │ add[0][0]         │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 150,   │        256 │ conv2d_3[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 20, 150,   │     36,928 │ batch_normalizat… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 20, 150,   │        256 │ conv2d_4[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 20, 150,   │          0 │ batch_normalizat… │
│                     │ 64)               │            │ add[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 10, 75,    │          0 │ add_1[0][0]       │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout (Dropout)   │ (None, 10, 75,    │          0 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 10, 75,    │     36,928 │ dropout[0][0]     │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 10, 75,    │        256 │ conv2d_5[0][0]  

 Total params: 1,814,145 (6.92 MB)

 Trainable params: 1,812,993 (6.92 MB)

 Non-trainable params: 1,152 (4.50 KB)


--- Step 5: Starting Model Training ---

Epoch 1/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 804ms/step - accuracy: 0.5476 - loss: 14.8753
Epoch 1: val_accuracy improved from -inf to 0.33333, saving model to testing.keras
4/4 ━━━━━━━━━━━━━━━━━━━━ 9s 1s/step - accuracy: 0.5418 - loss: 15.6352 - val_accuracy: 0.3333 - val_loss: 7.6760 - learning_rate: 0.0010
Epoch 2/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 812ms/step - accuracy: 0.6212 - loss: 10.3510
Epoch 2: val_accuracy improved from 0.33333 to 0.66667, saving model to testing.keras
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 937ms/step - accuracy: 0.6211 - loss: 10.1292 - val_accuracy: 0.6667 - val_loss: 6.4421 - learning_rate: 0.0010
Epoch 3/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 812ms/step - accuracy: 0.6400 - loss: 12.1422
Epoch 3: val_accuracy did not improve from 0.66667
4/4 ━━━━━━━━━━━━━━━━━━━━ 4s 892ms/step - accuracy: 0.6324 - loss: 12.4564 - val_accuracy: 0.6111 - val_loss: 4.2831 - learning_rate: 0.0010
Epoch 4/25
4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 833ms/step - accuracy: 0.67

training 2.0

In [ ]:
# # =========================================================================
# #
# #       Universal Expert Model Trainer (Definitive, Corrected Version)
# #
# # This script contains the corrected GroupNormalization layer to fix the
# # 'TypeError: unexpected keyword argument 'n'' bug.
# #
# # =========================================================================

# import os
# import sys
# import numpy as np
# import pandas as pd
# import librosa
# import librosa.effects
# import tensorflow as tf
# from sklearn.model_selection import train_test_split
# from tensorflow.keras.models import Sequential
# from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, Flatten, Dense, Layer
# from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
# from tensorflow.keras.regularizers import l2
# import random

# # ===============================================================
# #                !!! TASK CONFIGURATION !!!
# # ===============================================================
# # Run this script 4 times, changing these two variables each time.
# # Example for the first run:
# CLASSES_TO_TRAIN = ['COPD1', 'COPD2']
# MODEL_TO_SAVE_AS = "testing_1_2.keras"
# # ===============================================================

# # --- General Configuration ---
# LABEL_PATH = "/home/punith/Desktop/cHEAL 2.o/Labels.xlsx"
# AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/RespiratoryDatabase@TR"
# SAVE_DIR = "/home/punith/Desktop/cHEAL 2.o/All models"

# N_MELS, MAX_LEN, EPOCHS, BATCH_SIZE = 20, 150, 30, 32
# INITIAL_LEARNING_RATE = 0.0001


# # --- Step 2: Helper Functions & CORRECTED Custom Layer ---
# def augment(y, sr):
#     """Applies a random augmentation to the audio signal."""
#     if random.random() < 0.5:
#         rate = random.uniform(0.9, 1.1)
#         return librosa.effects.time_stretch(y, rate=rate)
#     else:
#         steps = random.randint(-2, 2)
#         return librosa.effects.pitch_shift(y, sr=sr, n_steps=steps)

# def extract_log_mel_spectrogram(path, do_augment=False):
#     """Loads an audio file and converts it to a Log-Mel Spectrogram."""
#     y, sr = librosa.load(path, sr=None)
#     if do_augment:
#         y = augment(y, sr)
    
#     mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
#     log_mel = librosa.power_to_db(mel_spec)
    
#     if log_mel.shape[1] < MAX_LEN:
#         log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
#     else:
#         log_mel = log_mel[:, :MAX_LEN]
#     return log_mel

# # --- CORRECTED GroupNormalization Layer ---
# class GroupNormalization(Layer):
#     """Custom Group Normalization layer with correct arguments."""
#     def __init__(self, groups=4, epsilon=1e-5, **kwargs):
#         super(GroupNormalization, self).__init__(**kwargs)
#         self.groups = groups
#         self.epsilon = epsilon
        
#     def build(self, input_shape):
#         dim = input_shape[-1]
#         # Using the full, correct keyword arguments: 'name', 'shape', 'initializer'
#         self.gamma = self.add_weight(name='gamma', shape=(1,1,1,dim), initializer='ones')
#         self.beta = self.add_weight(name='beta', shape=(1,1,1,dim), initializer='zeros')

#     def call(self, inputs):
#         input_shape = tf.shape(inputs)
#         N, H, W, C = input_shape[0], input_shape[1], input_shape[2], input_shape[3]
#         group_size = C // self.groups
#         reshaped = tf.reshape(inputs, [N, H, W, self.groups, group_size])
#         mean, var = tf.nn.moments(reshaped, [1, 2, 4], keepdims=True)
#         normalized = (reshaped - mean) / tf.sqrt(var + self.epsilon)
#         return tf.reshape(normalized, input_shape) * self.gamma + self.beta


# # --- Step 3: Main Training Logic ---
# TASK_NAME = f"{CLASSES_TO_TRAIN[0]}_vs_{CLASSES_TO_TRAIN[1]}"
# print(f"\n{'='*20} STARTING TRAINING FOR: {TASK_NAME} {'='*20}")

# df = pd.read_excel(LABEL_PATH)
# df_task = df[df["Diagnosis"].isin(CLASSES_TO_TRAIN)].copy()
# if df_task['Diagnosis'].nunique() < 2: raise ValueError(f"Not enough classes found for task {TASK_NAME}")

# df_task['label'] = df_task['Diagnosis'].apply(lambda x: 0 if x == CLASSES_TO_TRAIN[0] else 1)
# label_dict = dict(zip(df_task["Patient ID"], df_task["label"]))

# # Undersampling
# df_class0 = df_task[df_task['label'] == 0]
# df_class1 = df_task[df_task['label'] == 1]
# min_size = min(len(df_class0), len(df_class1))
# df_balanced = pd.concat([df_class0.sample(n=min_size, random_state=42), df_class1.sample(n=min_size, random_state=42)])
# patient_ids_balanced, patient_labels_balanced = list(df_balanced["Patient ID"]), list(df_balanced["label"])

# train_pids, test_pids, _, _ = train_test_split(patient_ids_balanced, patient_labels_balanced, test_size=0.25, random_state=42, stratify=patient_labels_balanced)

# # Data Generation
# X_train, y_train, X_test, y_test = [], [], [], []
# for pid in patient_ids_balanced:
#     label = label_dict[pid]
#     paths = [os.path.join(AUDIO_DIR, f) for f in os.listdir(AUDIO_DIR) if f.startswith(str(pid)) and f.endswith('.wav')]
#     for path in paths:
#         try:
#             if pid in train_pids:
#                 X_train.append(extract_log_mel_spectrogram(path))
#                 y_train.append(label)
#                 for _ in range(3):
#                     X_train.append(extract_log_mel_spectrogram(path, do_augment=True))
#                     y_train.append(label)
#             elif pid in test_pids:
#                 X_test.append(extract_log_mel_spectrogram(path))
#                 y_test.append(label)
#         except Exception as e: print(f"Warning on {path}: {e}")

# # Normalization
# X_train_raw = np.array(X_train)
# mean_val, std_val = np.mean(X_train_raw, axis=0), np.std(X_train_raw, axis=0)
# model_name = MODEL_TO_SAVE_AS.replace('.keras', '')
# mean_path = os.path.join(SAVE_DIR, f"logmel_{model_name}_mean.npy")
# std_path = os.path.join(SAVE_DIR, f"logmel_{model_name}_std.npy")
# print(f"\nSaving normalization stats to {mean_path}...")
# np.save(mean_path, mean_val); np.save(std_path, std_val)

# X_train = (X_train_raw - mean_val) / (std_val + 1e-6)
# X_test = (np.array(X_test) - mean_val) / (std_val + 1e-6)
# X_train, X_test = X_train[..., np.newaxis], X_test[..., np.newaxis]
# y_train, y_test = np.array(y_train), np.array(y_test)

# print(f"Final Shapes - X_train: {X_train.shape}, X_test: {X_test.shape}")

# # Model Architecture
# model = Sequential([
#     Input(shape=(N_MELS, MAX_LEN, 1)),
#     Conv2D(16, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
#     GroupNormalization(), MaxPooling2D(), Dropout(0.3),
#     Conv2D(32, (3, 3), activation='gelu', padding='same', kernel_regularizer=l2(0.001)),
#     GroupNormalization(), MaxPooling2D(), Dropout(0.3),
#     Flatten(), Dense(64, activation='gelu', kernel_regularizer=l2(0.001)),
#     Dropout(0.5), Dense(1, activation='sigmoid')
# ])
# model.compile(optimizer=tf.keras.optimizers.Adam(INITIAL_LEARNING_RATE), loss='binary_crossentropy', metrics=['accuracy'])
# model.summary()

# # Training
# print("\nStarting training...")
# model_path = os.path.join(SAVE_DIR, MODEL_TO_SAVE_AS)
# callbacks = [EarlyStopping('val_loss', patience=10, restore_best_weights=True), ModelCheckpoint(model_path, save_best_only=True), ReduceLROnPlateau(patience=4)]
# model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=EPOCHS, batch_size=BATCH_SIZE, callbacks=callbacks)

# print(f"\n{'='*20} FINISHED TRAINING: {MODEL_TO_SAVE_AS} {'='*20}")


==================== STARTING TRAINING FOR: COPD1_vs_COPD2 ====================

Saving normalization stats to /home/punith/Desktop/cHEAL 2.o/All models/logmel_testing_1_2_mean.npy...
Final Shapes - X_train: (336, 20, 150, 1), X_test: (36, 20, 150, 1)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_20 (Conv2D)              │ (None, 20, 150, 16)    │           160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_2           │ (None, 20, 150, 16)    │            32 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_6 (MaxPooling2D)  │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 10, 75, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_21 (Conv2D)              │ (None, 10, 75, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ group_normalization_3           │ (None, 10, 75, 32)     │            64 │
│ (GroupNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_7 (MaxPooling2D)  │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_10 (Dropout)            │ (None, 5, 37, 32)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 5920)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       378,944 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_11 (Dropout)            │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 383,905 (1.46 MB)

 Trainable params: 383,905 (1.46 MB)

 Non-trainable params: 0 (0.00 B)


Starting training...
Epoch 1/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 3s 108ms/step - accuracy: 0.4889 - loss: 1.7539 - val_accuracy: 0.4167 - val_loss: 1.2367 - learning_rate: 1.0000e-04
Epoch 2/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 78ms/step - accuracy: 0.6608 - loss: 0.9313 - val_accuracy: 0.3889 - val_loss: 1.4312 - learning_rate: 1.0000e-04
Epoch 3/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 1s 85ms/step - accuracy: 0.7254 - loss: 0.8234 - val_accuracy: 0.4444 - val_loss: 1.4664 - learning_rate: 1.0000e-04
Epoch 4/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 152ms/step - accuracy: 0.7387 - loss: 0.6741 - val_accuracy: 0.4444 - val_loss: 1.3944 - learning_rate: 1.0000e-04
Epoch 5/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 166ms/step - accuracy: 0.7988 - loss: 0.5349 - val_accuracy: 0.3056 - val_loss: 1.4570 - learning_rate: 1.0000e-04
Epoch 6/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 177ms/step - accuracy: 0.7920 - loss: 0.5834 - val_accuracy: 0.3056 - val_loss: 1.4524 - learning_rate: 1.0000e-05
Epoch 7/30
11/11 ━━━━━━━━━━━━━━━━━━━━ 2s 179

In [3]:
# =========================================================================
#
#       Inference Script for "testing.keras"
#
# This script loads the "testing.keras" model and its specific
# normalization stats to predict the progression of a single audio file
# between Stage 1 and Stage 2.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "testing.keras" training run ---
MODEL_PATH = "testing.keras"
MEAN_PATH = "logmel_testing_mean.npy"
STD_PATH = "logmel_testing_std.npy"

# --- The audio file you want to predict ---
# !!! UPDATE THIS PATH TO YOUR AUDIO FILE !!!
AUDIO_FILE_TO_PREDICT = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/142_1b1_Pl_mc_LittC2SE.wav"
# Example: "/home/punith/Desktop/new_data/patient_X_L1.wav"
# ---------------------------------------------

# --- Parameters that MUST match the training script ---
N_MELS = 128
MAX_LEN = 150
LABELS = {0: 'COPD1', 1: 'COPD2'} # Model was trained with COPD1=0, COPD2=1

def predict_progression_stage(audio_path):
    """
    Loads the trained model, processes a single audio file into a
    Log-Mel Spectrogram, and returns the predicted progression stage.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading progression model ('testing.keras') and normalization stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False) # compile=False for faster loading
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        print("Ensure 'testing.keras', 'logmel_testing_mean.npy', and 'logmel_testing_std.npy' are present.")
        return

    # --- 2. Process the New Audio File ---
    print(f"\n--- Processing audio file: {os.path.basename(audio_path)} ---")
    try:
        y, sr = librosa.load(audio_path, sr=None)
        
        mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
        log_mel = librosa.power_to_db(mel_spec)
        
        if log_mel.shape[1] < MAX_LEN:
            log_mel = np.pad(log_mel, ((0, 0), (0, MAX_LEN - log_mel.shape[1])), mode='constant')
        else:
            log_mel = log_mel[:, :MAX_LEN]
            
        log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
        log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
        
    except Exception as e:
        print(f"❌ ERROR: Failed to process the audio file. Error: {e}")
        return

    # --- 3. Make the Prediction ---
    print("\n--- Making prediction... ---")
    try:
        prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
        
        if prediction_prob >= 0.5:
            predicted_class_name = LABELS[1] # 'COPD2'
            confidence = prediction_prob
        else:
            predicted_class_name = LABELS[0] # 'COPD1'
            confidence = 1 - prediction_prob # Confidence in the 'COPD1' prediction

    except Exception as e:
        print(f"❌ ERROR: Model failed to predict. Error: {e}")
        return

    # --- 4. Display the Result ---
    print("\n" + "="*50)
    print("--- 🩺 Progression Prediction Result ---")
    print(f"The model predicts the patient's stage for this recording is:")
    print(f"    >> {predicted_class_name} <<")
    print(f"    Confidence: {confidence * 100:.2f}%")
    print("="*50)
    print(f"Technical Details:")
    print(f"  - Raw Model Output (0.0 ≈ COPD1, 1.0 ≈ COPD2): {prediction_prob:.4f}")

# --- Run the prediction function ---
if __name__ == "__main__":
    if os.path.exists(AUDIO_FILE_TO_PREDICT):
        predict_progression_stage(AUDIO_FILE_TO_PREDICT)
    else:
        print(f"❌ ERROR: The specified audio file does not exist at '{AUDIO_FILE_TO_PREDICT}'")

--- Loading progression model ('testing.keras') and normalization stats... ---
✅ Model and stats loaded successfully.

--- Processing audio file: 142_1b1_Pl_mc_LittC2SE.wav ---

--- Making prediction... ---

--- 🩺 Progression Prediction Result ---
The model predicts the patient's stage for this recording is:
    >> COPD2 <<
    Confidence: 100.00%
Technical Details:
  - Raw Model Output (0.0 ≈ COPD1, 1.0 ≈ COPD2): 1.0000


114

In [5]:
# =========================================================================
#
#       Patient Analysis Script for "testing.keras" (COPD 1 vs 2)
#
# This script loads the trained binary model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 1 to Stage 2.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "testing.keras" (1 vs 2) training run ---
MODEL_PATH = "testing.keras"
MEAN_PATH = "logmel_testing_mean.npy" # <-- This used N_MELS = 128
STD_PATH = "logmel_testing_std.npy"   # <-- This used N_MELS = 128

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "114"

# --- Parameters that MUST match the training script ---
N_MELS = 128 # Must be 128 to match the model that was trained
MAX_LEN = 150
LABELS = {0: 'COPD1', 1: 'COPD2'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 1-vs-2 (N_MELS=128) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            # a. Process the audio file
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            # b. Make prediction
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (1 vs 2) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 1-vs-2 (N_MELS=128) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 114 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 5 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 114_1b4_Al_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 114_1b4_Ar_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 114_1b4_Lr_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 114_1b4_Pl_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 114_1b4_Pr_mc_AKGC417L.wav               | Predicted: COPD2    | Confidence: 100.00%

--- 📋 Final Analysis Summary for Patient 114 (1 vs 2) ---
Based on a majority vote of all 5 recordings, the most likely stage is:
    >> COPD2 <<
    (This received 5 out of 5 

142

In [4]:
# =========================================================================
#
#       Patient Analysis Script for "testing.keras" (COPD 1 vs 2)
#
# This script loads the trained binary model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 1 to Stage 2.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "testing.keras" (1 vs 2) training run ---
MODEL_PATH = "testing.keras"
MEAN_PATH = "logmel_testing_mean.npy" # <-- This used N_MELS = 128
STD_PATH = "logmel_testing_std.npy"   # <-- This used N_MELS = 128

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "142"

# --- Parameters that MUST match the training script ---
N_MELS = 128 # Must be 128 to match the model that was trained
MAX_LEN = 150
LABELS = {0: 'COPD1', 1: 'COPD2'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 1-vs-2 (N_MELS=128) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            # a. Process the audio file
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            # b. Make prediction
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (1 vs 2) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 1-vs-2 (N_MELS=128) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 142 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 1 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 142_1b1_Pl_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%

--- 📋 Final Analysis Summary for Patient 142 (1 vs 2) ---
Based on a majority vote of all 1 recordings, the most likely stage is:
    >> COPD2 <<
    (This received 1 out of 1 votes, for a 100.00% consensus.)


110

In [6]:
# =========================================================================
#
#       Patient Analysis Script for "testing.keras" (COPD 1 vs 2)
#
# This script loads the trained binary model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 1 to Stage 2.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "testing.keras" (1 vs 2) training run ---
MODEL_PATH = "testing.keras"
MEAN_PATH = "logmel_testing_mean.npy" # <-- This used N_MELS = 128
STD_PATH = "logmel_testing_std.npy"   # <-- This used N_MELS = 128

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "110"

# --- Parameters that MUST match the training script ---
N_MELS = 128 # Must be 128 to match the model that was trained
MAX_LEN = 150
LABELS = {0: 'COPD1', 1: 'COPD2'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 1-vs-2 (N_MELS=128) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            # a. Process the audio file
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            # b. Make prediction
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (1 vs 2) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 1-vs-2 (N_MELS=128) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 110 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 5 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 110_1b1_Pr_sc_Meditron.wav               | Predicted: COPD1    | Confidence: 100.00%
  -> File: 110_1p1_Al_sc_Meditron.wav               | Predicted: COPD1    | Confidence: 100.00%
  -> File: 110_1p1_Ll_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 110_1p1_Lr_sc_Meditron.wav               | Predicted: COPD1    | Confidence: 100.00%
  -> File: 110_1p1_Pr_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%

--- 📋 Final Analysis Summary for Patient 110 (1 vs 2) ---
Based on a majority vote of all 5 recordings, the most likely stage is:
    >> COPD1 <<
    (This received 3 out of 5 

106

In [7]:
# =========================================================================
#
#       Patient Analysis Script for "testing.keras" (COPD 1 vs 2)
#
# This script loads the trained binary model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 1 to Stage 2.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "testing.keras" (1 vs 2) training run ---
MODEL_PATH = "testing.keras"
MEAN_PATH = "logmel_testing_mean.npy" # <-- This used N_MELS = 128
STD_PATH = "logmel_testing_std.npy"   # <-- This used N_MELS = 128

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "106"

# --- Parameters that MUST match the training script ---
N_MELS = 128 # Must be 128 to match the model that was trained
MAX_LEN = 150
LABELS = {0: 'COPD1', 1: 'COPD2'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 1-vs-2 (N_MELS=128) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            # a. Process the audio file
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            # b. Make prediction
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (1 vs 2) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 1-vs-2 (N_MELS=128) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 106 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 2 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 106_2b1_Pl_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 106_2b1_Pr_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%

--- 📋 Final Analysis Summary for Patient 106 (1 vs 2) ---
Based on a majority vote of all 2 recordings, the most likely stage is:
    >> COPD2 <<
    (This received 2 out of 2 votes, for a 100.00% consensus.)


221

In [8]:
# =========================================================================
#
#       Patient Analysis Script for "testing.keras" (COPD 1 vs 2)
#
# This script loads the trained binary model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 1 to Stage 2.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "testing.keras" (1 vs 2) training run ---
MODEL_PATH = "testing.keras"
MEAN_PATH = "logmel_testing_mean.npy" # <-- This used N_MELS = 128
STD_PATH = "logmel_testing_std.npy"   # <-- This used N_MELS = 128

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "221"

# --- Parameters that MUST match the training script ---
N_MELS = 128 # Must be 128 to match the model that was trained
MAX_LEN = 150
LABELS = {0: 'COPD1', 1: 'COPD2'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 1-vs-2 (N_MELS=128) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            # a. Process the audio file
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            # b. Make prediction
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (1 vs 2) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 1-vs-2 (N_MELS=128) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 221 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 12 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 221_2b1_Al_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 221_2b1_Ar_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 221_2b1_Lr_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 221_2b1_Pl_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 221_2b2_Al_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 221_2b2_Ar_mc_LittC2SE.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 221_2b2_Lr_mc_LittC2SE.wav               | Predicted: COPD2    | Conf

222

In [10]:
# =========================================================================
#
#       Patient Analysis Script for "testing.keras" (COPD 1 vs 2)
#
# This script loads the trained binary model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 1 to Stage 2.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "testing.keras" (1 vs 2) training run ---
MODEL_PATH = "testing.keras"
MEAN_PATH = "logmel_testing_mean.npy" # <-- This used N_MELS = 128
STD_PATH = "logmel_testing_std.npy"   # <-- This used N_MELS = 128

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "222"

# --- Parameters that MUST match the training script ---
N_MELS = 128 # Must be 128 to match the model that was trained
MAX_LEN = 150
LABELS = {0: 'COPD1', 1: 'COPD2'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 1-vs-2 (N_MELS=128) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            # a. Process the audio file
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            # b. Make prediction
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (1 vs 2) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 1-vs-2 (N_MELS=128) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 222 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 3 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 222_1b1_Ar_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 93.69%
  -> File: 222_1b1_Lr_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 222_1b1_Pr_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%

--- 📋 Final Analysis Summary for Patient 222 (1 vs 2) ---
Based on a majority vote of all 3 recordings, the most likely stage is:
    >> COPD2 <<
    (This received 3 out of 3 votes, for a 100.00% consensus.)


223

In [11]:
# =========================================================================
#
#       Patient Analysis Script for "testing.keras" (COPD 1 vs 2)
#
# This script loads the trained binary model and analyzes all recordings
# for a specific patient to determine if they show signs of progression
# from Stage 1 to Stage 2.
#
# =========================================================================

import os
import librosa
import numpy as np
import pandas as pd
from tensorflow.keras.models import load_model
import warnings

# Suppress librosa warnings
warnings.filterwarnings('ignore', category=UserWarning)

# --- Step 1: Configuration ---
# --- Files created by your "testing.keras" (1 vs 2) training run ---
MODEL_PATH = "testing.keras"
MEAN_PATH = "logmel_testing_mean.npy" # <-- This used N_MELS = 128
STD_PATH = "logmel_testing_std.npy"   # <-- This used N_MELS = 128

# --- The folder containing ALL new audio files for prediction ---
# !!! UPDATE THIS PATH TO YOUR NEW DATASET FOLDER !!!
NEW_AUDIO_DIR = "/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/"

# --- The Patient ID you want to analyze ---
# !!! UPDATE THIS TO THE PATIENT YOU WANT TO TEST (e.g., '142') !!!
PATIENT_ID_TO_ANALYZE = "223"

# --- Parameters that MUST match the training script ---
N_MELS = 128 # Must be 128 to match the model that was trained
MAX_LEN = 150
LABELS = {0: 'COPD1', 1: 'COPD2'}


# --- Main Analysis Function ---
def analyze_patient_recordings(patient_id, audio_dir):
    """
    Finds all audio files for a given patient ID, processes each one,
    and reports the model's prediction for each.
    """
    # --- 1. Load Model and Normalization Statistics ---
    print("--- Loading 1-vs-2 (N_MELS=128) model and stats... ---")
    try:
        model = load_model(MODEL_PATH, compile=False)
        training_mean = np.load(MEAN_PATH)
        training_std = np.load(STD_PATH)
        print("✅ Model and stats loaded successfully.")
    except Exception as e:
        print(f"❌ FATAL ERROR: Could not load required files. Error: {e}")
        return

    # --- 2. Find all audio files for the specified patient ---
    print(f"\n--- Searching for recordings for Patient ID: {patient_id} in '{audio_dir}' ---")
    
    patient_files = [os.path.join(audio_dir, f) for f in os.listdir(audio_dir)
                     if f.startswith(str(patient_id) + '_') and f.endswith('.wav')]
    
    if not patient_files:
        print(f"❌ No .wav files found for Patient ID '{patient_id}'.")
        return

    print(f"Found {len(patient_files)} audio files to analyze.")
    
    # --- 3. Loop through each file and predict ---
    print("\n--- 🩺 Individual Recording Predictions ---")
    all_predictions_idx = []
    
    for file_path in sorted(patient_files):
        try:
            # a. Process the audio file
            y, sr = librosa.load(file_path, sr=None)
            mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_mels=N_MELS)
            log_mel = librosa.power_to_db(mel_spec)
            if log_mel.shape[1] < MAX_LEN: log_mel = np.pad(log_mel, ((0,0), (0, MAX_LEN - log_mel.shape[1])))
            else: log_mel = log_mel[:, :MAX_LEN]
            log_mel_normalized = (log_mel - training_mean) / (training_std + 1e-6)
            log_mel_final = log_mel_normalized[np.newaxis, ..., np.newaxis]
            
            # b. Make prediction
            prediction_prob = model.predict(log_mel_final, verbose=0)[0][0]
            
            if prediction_prob >= 0.5:
                predicted_idx, predicted_name, confidence = 1, LABELS[1], prediction_prob
            else:
                predicted_idx, predicted_name, confidence = 0, LABELS[0], 1 - prediction_prob
                
            all_predictions_idx.append(predicted_idx)
            print(f"  -> File: {os.path.basename(file_path):<40} | Predicted: {predicted_name:<8} | Confidence: {confidence*100:.2f}%")
            
        except Exception as e:
            print(f"  -> ERROR processing file {os.path.basename(file_path)}: {e}")

    # --- 4. Provide a final summary for the patient based on majority vote ---
    if all_predictions_idx:
        final_prediction_idx = max(set(all_predictions_idx), key=all_predictions_idx.count)
        final_verdict = LABELS[final_prediction_idx]
        vote_count = all_predictions_idx.count(final_prediction_idx)
        verdict_confidence = (vote_count / len(all_predictions_idx)) * 100
        
        print("\n" + "="*60)
        print(f"--- 📋 Final Analysis Summary for Patient {patient_id} (1 vs 2) ---")
        print(f"Based on a majority vote of all {len(all_predictions_idx)} recordings, the most likely stage is:")
        print(f"    >> {final_verdict} <<")
        print(f"    (This received {vote_count} out of {len(all_predictions_idx)} votes, for a {verdict_confidence:.2f}% consensus.)")
        print("="*60)

# --- Run the main analysis function ---
if __name__ == "__main__":
    analyze_patient_recordings(PATIENT_ID_TO_ANALYZE, NEW_AUDIO_DIR)

--- Loading 1-vs-2 (N_MELS=128) model and stats... ---
✅ Model and stats loaded successfully.

--- Searching for recordings for Patient ID: 223 in '/home/punith/Desktop/cHEAL 2.o/archive/Respiratory_Sound_Database/Respiratory_Sound_Database/audio_and_txt_files/' ---
Found 6 audio files to analyze.

--- 🩺 Individual Recording Predictions ---
  -> File: 223_1b1_Al_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 223_1b1_Ar_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 223_1b1_Ll_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 223_1b1_Lr_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 223_1b1_Pl_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%
  -> File: 223_1b1_Pr_sc_Meditron.wav               | Predicted: COPD2    | Confidence: 100.00%

--- 📋 Final Analysis Summary for Patient 223 (1 vs 2) ---
Based on a majority vo